# Density & BI-RADS–Aware Triage Method (DB-ATRG) — Colab (CPU Runtime)

Simulates clinical priority worklist re-ordering from model-generated predictions:
1. **Phase I (Density Flagging)**: Identifies **ACR D** (extremely dense breast tissue) cases prone to masking bias and routes them for expedited/supplemental screening.
2. **Phase II (Urgency Ranking)**: Orders remaining cases by cumulative urgency score:
   $$S = \sum_{i=1}^{N} (B_i)^k + D$$
   where $B_i$ is the BI-RADS category of each finding, $k=2$, and $D$ is the ACR density ($A=1, B=2, C=3$).
3. **Evaluation**: Compares the Treated Priority Queue against the Random FIFO Queue baseline, computing high-risk capture rates (BI-RADS 4/5 in top 10%), average rank shift, and generating a 5-panel publication-ready figure.

Input: `ROOT_DIR/results/eval_predictions.jsonl` produced by `medgemma-train-eval.ipynb`.

In [ ]:
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print('Running outside Colab environment.')

In [ ]:
# Directory configuration matching the pipeline layout
if IS_COLAB:
    ROOT_DIR = Path('/content/drive/MyDrive/MedGemma2026/main')
else:
    # If run locally in repo
    ROOT_DIR = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()

RESULTS_DIR = ROOT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_JSONL_PATH = RESULTS_DIR / 'eval_predictions.jsonl'

print(f'ROOT_DIR:           {ROOT_DIR}')
print(f'RESULTS_DIR:        {RESULTS_DIR}')
print(f'Target Predictions: {DEFAULT_JSONL_PATH}')

In [ ]:
BIRADS_K = 2
ACR_DENSITY_MAP = {'A': 1, 'B': 2, 'C': 3, 'D': 4}

EXCLUDED_BIRADS = {0, 6}

BIRADS_VALUES = [1, 2, 3, 4, 5]
BIRADS_WEIGHTS = [0.653, 0.25, 0.058, 0.023, 0.016]
BIRADS_WEIGHT_MAP = dict(zip(BIRADS_VALUES, BIRADS_WEIGHTS))

NUM_BIRADS_WEIGHTS = [0.75, 0.20, 0.05]

ACR_VALUES = ['A', 'B', 'C', 'D']
ACR_WEIGHTS = [0.075, 0.535, 0.34, 0.05]
ACR_WEIGHT_MAP = dict(zip(ACR_VALUES, ACR_WEIGHTS))

In [ ]:
@dataclass
class MammographyCase:
    case_id: int
    birads_list: list
    acr: str
    gt_max_birads: int = None

    @property
    def max_birads(self) -> int:
        return max(self.birads_list)

    def score(self) -> float:
        return sum(b**BIRADS_K for b in self.birads_list) + ACR_DENSITY_MAP[self.acr]


def apply_triage(cases: list[MammographyCase]) -> dict:
    phase1, phase2 = [], []

    for case in cases:
        if case.acr == 'D':
            phase1.append(case)
        else:
            phase2.append(case)

    return {'phase1': phase1, 'phase2': sorted(phase2, key=lambda c: c.score(), reverse=True)}

In [ ]:
def load_cases_from_jsonl(
    jsonl_path: str = None,
    k: int = 100,
    seed: int = 2,
    filter_correct: bool = True,
    weight_mode: str = 'additive',
) -> list[MammographyCase]:
    if jsonl_path is None:
        if DEFAULT_JSONL_PATH.exists():
            jsonl_path = str(DEFAULT_JSONL_PATH)
        else:
            local_fallback = Path('results') / 'eval_predictions.jsonl'
            jsonl_path = str(local_fallback)

    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(
            f'Predictions file not found at: {jsonl_path}. '
            f'Make sure medgemma-train-eval.ipynb has run and saved eval_predictions.jsonl into results folder.'
        )

    all_cases = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            case_id = item.get('i', len(all_cases))
            text = item.get('text', '')

            # Extract ACR density from text prediction or fallback to acr.pred
            acr_match = re.search(r'ACR\s*([A-D])', text)
            if acr_match:
                acr = acr_match.group(1)
            elif 'acr' in item and isinstance(item['acr'], dict) and 'pred' in item['acr']:
                acr = item['acr']['pred']
            else:
                acr = 'B'

            # Extract BI-RADS list strictly from the BI-RADS header line (e.g. 'BI-RADS: 4, 5')
            header_match = re.search(r'BI-RADS:\s*([^\n]+)', text)
            birads_list = []
            if header_match:
                digits = re.findall(r'\b([1-5])[a-c]?\b', header_match.group(1))
                birads_list.extend([int(d) for d in digits])

            if not birads_list:
                if 'birads' in item and isinstance(item['birads'], dict) and 'pred' in item['birads']:
                    birads_list = [int(item['birads']['pred'])]
                else:
                    birads_list = [1]

            # Ground truth max BI-RADS strictly from the BI-RADS header line in reference report
            ref = item.get('ref', '')
            ref_header = re.search(r'BI-RADS:\s*([^\n]+)', ref)
            if ref_header:
                ref_digits = re.findall(r'\b([1-5])[a-c]?\b', ref_header.group(1))
                gt_max_birads = max([int(d) for d in ref_digits]) if ref_digits else max(birads_list)
            else:
                gt_max_birads = max(birads_list)

            ref_acr_match = re.search(r'ACR\s*([A-D])', ref)
            gt_acr = ref_acr_match.group(1) if ref_acr_match else 'B'

            c = MammographyCase(
                case_id=case_id,
                birads_list=birads_list,
                acr=acr,
                gt_max_birads=gt_max_birads,
            )
            c.gt_acr = gt_acr
            all_cases.append(c)

    # Filter for correctly predicted cases if requested
    if filter_correct:
        all_cases = [c for c in all_cases if c.max_birads == c.gt_max_birads]

    if weight_mode == 'birads_only':
        case_weights = [BIRADS_WEIGHT_MAP.get(c.max_birads, 0.016) for c in all_cases]
    elif weight_mode == 'additive':
        case_weights = [
            BIRADS_WEIGHT_MAP.get(c.max_birads, 0.016) + ACR_WEIGHT_MAP.get(c.acr, 0.05)
            for c in all_cases
        ]
    else:  # product
        case_weights = [
            BIRADS_WEIGHT_MAP.get(c.max_birads, 0.016) * ACR_WEIGHT_MAP.get(c.acr, 0.05)
            for c in all_cases
        ]

    if k is not None and k < len(all_cases):
        case_weights_arr = np.array(case_weights, dtype=float)
        prob = case_weights_arr / case_weights_arr.sum()
        rng = np.random.default_rng(seed)
        sampled_indices = rng.choice(len(all_cases), size=k, replace=False, p=prob)
        return [all_cases[idx] for idx in sampled_indices]

    return all_cases


def generate_cases(k: int = 100, seed: int = 2) -> list[MammographyCase]:
    rng = random.Random(seed)
    cases = []

    for i in range(k):
        n_findings = rng.choices([1, 2, 3], weights=NUM_BIRADS_WEIGHTS)[0]
        birads_list = rng.choices(BIRADS_VALUES, weights=BIRADS_WEIGHTS, k=n_findings)
        acr = rng.choices(ACR_VALUES, weights=ACR_WEIGHTS)[0]
        cases.append(MammographyCase(case_id=i + 1, birads_list=birads_list, acr=acr))

    return cases

In [ ]:
def build_dataframe(cases, label: str, compute_score: bool = True) -> pd.DataFrame:
    rows = []
    for rank, case in enumerate(cases, start=1):
        row = {
            f'{label}_rank': rank,
            'case_id': case.case_id,
            'birads_list': str(case.birads_list),
            'max_birads': case.max_birads,
            'acr': case.acr,
        }

        if hasattr(case, 'gt_max_birads') and case.gt_max_birads is not None:
            row['gt_max_birads'] = case.gt_max_birads

        if compute_score:
            row['score'] = case.score()

        rows.append(row)

    return pd.DataFrame(rows)


def rank_shift_stats(df_random: pd.DataFrame, df_treated: pd.DataFrame) -> pd.DataFrame:
    extra_cols = ['TREATED_rank']
    if 'gt_max_birads' in df_treated.columns:
        extra_cols.append('gt_max_birads')
    merged = df_random.merge(df_treated[['case_id'] + extra_cols], on='case_id').rename(
        columns={'RANDOM_rank': 'random_rank', 'TREATED_rank': 'treated_rank'}
    )
    # Positive rank shift means PROMOTED (e.g., Random Rank 100 - Treated Rank 10 = +90 shift)
    merged['rank_shift'] = merged['random_rank'] - merged['treated_rank']
    return merged


def compute_metrics(merged: pd.DataFrame, triage: dict) -> dict:
    k_phase2 = len(merged)
    top_10_pct = max(1, int(k_phase2 * 0.10))

    if 'gt_max_birads' in merged.columns:
        merged['is_high_risk'] = merged['gt_max_birads'] >= 4
    else:
        merged['is_high_risk'] = merged['max_birads'] >= 4

    total_high_risk = merged['is_high_risk'].sum()

    top_treated = merged.nsmallest(top_10_pct, 'treated_rank')
    top_random = merged.nsmallest(top_10_pct, 'random_rank')

    treated_capture = top_treated['is_high_risk'].sum() / total_high_risk if total_high_risk else 0
    random_capture = top_random['is_high_risk'].sum() / total_high_risk if total_high_risk else 0

    return {
        'Phase II Scored Cases': k_phase2,
        'Total BIRADS 4/5 Cases': int(total_high_risk),
        'Avg Score (Top 10% TREATED)': round(top_treated['score'].mean(), 2),
        'Avg Score (Top 10% RANDOM)': round(top_random['score'].mean(), 2),
        'BIRADS 4/5 Found in Top 10% (TREATED)': f'{treated_capture:.1%}',
        'BIRADS 4/5 Found in Top 10% (RANDOM)': f'{random_capture:.1%}',
        'Mean Rank of BIRADS 4/5 (TREATED)': round(
            merged[merged['is_high_risk']]['treated_rank'].mean(), 1
        ),
        'Mean Rank of BIRADS 4/5 (RANDOM)': round(
            merged[merged['is_high_risk']]['random_rank'].mean(), 1
        ),
    }

In [ ]:
def plot_analysis(merged: pd.DataFrame, triage: dict, output_path: str = 'analysis.png'):
    k_phase2 = len(merged)
    merged['is_high_risk'] = merged['max_birads'] >= 4
    metrics = compute_metrics(merged, triage)

    fig = plt.figure(figsize=(18, 12))
    fig.subplots_adjust(top=0.95, bottom=0.22, left=0.06, right=0.94)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.25)

    colors = {
        'phase2': '#08306b',   # Blue
        'random': '#f3b86b',   # Orange
        'boxplot': '#cbd5e1',  # Neutral light gray/blue
        'high_risk': '#4a5568', # Neutral dark gray
    }

    # ── 1. Score vs max BIRADS ──
    ax1 = fig.add_subplot(gs[0, 0])
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(merged))
    ax1.scatter(
        merged['max_birads'] + jitter, merged['score'], c=merged['max_birads'], cmap='plasma', alpha=0.6, s=20
    )
    ax1.set_xlabel('Max BIRADS', fontsize=14)
    ax1.set_ylabel('Assigned Urgency Score', fontsize=14)
    ax1.tick_params(axis='both', which='major', labelsize=12)

    # ── 2. Queue Position of High-Risk Cases ──
    ax2 = fig.add_subplot(gs[0, 1])
    high_risk_cases = merged[merged['is_high_risk']]

    # Plotting histograms of where high-risk cases ended up in the queue
    ax2.hist(
        high_risk_cases['random_rank'],
        bins=20,
        alpha=0.5,
        color=colors['random'],
        density=True,
    )
    ax2.hist(
        high_risk_cases['treated_rank'],
        bins=20,
        alpha=0.7,
        color=colors['phase2'],
        density=True,
    )

    ax2.set_xlabel('Queue Position', fontsize=14)
    ax2.set_ylabel('Density of High-Risk Cases', fontsize=14)
    ax2.set_xlim(1, k_phase2)
    ax2.tick_params(axis='both', which='major', labelsize=12)

    # ── 3. Rank shift by Max BIRADS (Boxplot) ──
    ax3 = fig.add_subplot(gs[0, 2])
    birads_unique = sorted(merged['max_birads'].unique())
    shift_data = [merged[merged['max_birads'] == b]['rank_shift'].values for b in birads_unique]

    bplot = ax3.boxplot(
        shift_data, tick_labels=birads_unique, patch_artist=True, medianprops=dict(color='black')
    )
    for patch in bplot['boxes']:
        patch.set_facecolor(colors['boxplot'])

    ax3.axhline(0, color=colors['high_risk'], linestyle='--', linewidth=1.3)
    ax3.set_xlabel('Max BIRADS', fontsize=14)
    ax3.set_ylabel('Rank Shift', fontsize=14)
    ax3.tick_params(axis='both', which='major', labelsize=12)

    # ── 4. Cumulative High-Risk Capture ──
    ax4 = fig.add_subplot(gs[1, 0:2])
    treated_sorted = merged.sort_values('treated_rank')
    random_sorted = merged.sort_values('random_rank')
    x = np.arange(1, k_phase2 + 1)

    ax4.plot(
        x,
        treated_sorted['is_high_risk'].cumsum().values,
        color=colors['phase2'],
        linewidth=2.5,
    )
    ax4.plot(
        x,
        random_sorted['is_high_risk'].cumsum().values,
        color=colors['random'],
        linewidth=2,
        linestyle='--',
    )

    ax4.fill_between(
        x,
        random_sorted['is_high_risk'].cumsum().values,
        treated_sorted['is_high_risk'].cumsum().values,
        color=colors['phase2'],
        alpha=0.3,
    )

    ax4.set_xlabel('Total Cases Reviewed by Radiologist', fontsize=14)
    ax4.set_ylabel('Number of High-Risk Cases Found', fontsize=14)
    ax4.grid(alpha=0.3)
    ax4.set_xlim(1, k_phase2)
    ax4.tick_params(axis='both', which='major', labelsize=12)

    # ── 5. Metrics table ──
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis('off')
    table_data = [[str(k), str(v)] for k, v in metrics.items()]

    tbl = ax5.table(
        cellText=table_data,
        colLabels=['Performance Metric', 'Value'],
        loc='center',
        cellLoc='left',
        colWidths=[0.70, 0.30],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(12)
    tbl.scale(1, 1.8)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold')

        if col == 1:
            cell.set_text_props(ha='center')

    caption_text = (
        'Figure Caption:\n'
        'This multi-panel figure evaluates the mammography queue optimization algorithm. Blue colors represent the Treated Queue (#08306b, '
        'prioritized using the urgency score), while orange colors represent the Random Queue (#f3b86b, baseline control). All titles and legends '
        'have been removed from the panels and described here for clarity.\n'
        '• Panel 1 (Top Left): Assigned Urgency Score (y-axis) vs. Max BI-RADS score (x-axis) for algorithm validation. Urgency score increases with higher BI-RADS categories.\n'
        '• Panel 2 (Top Middle): Density of high-risk (BI-RADS 4/5) cases across queue positions. The Treated Queue (blue) groups high-risk cases at the front, while the Random Queue (orange) distributes them uniformly.\n'
        '• Panel 3 (Top Right): Rank shift of cases by Max BI-RADS category. A positive rank shift denotes case promotion. The horizontal dashed line (gray) indicates zero shift.\n'
        '• Panel 4 (Bottom Left/Middle): Cumulative discovery of high-risk cases as a function of the total cases reviewed by the radiologist. The Treated Queue (solid blue line) finds cases significantly faster than the Random Queue (dashed orange line). The light blue shaded region illustrates the Clinical Advantage Gap.\n'
        '• Panel 5 (Bottom Right): Performance metrics table summarizing key statistics, including queue sizes, average urgency score, capture rates in the top 10%, and mean ranks.'
    )
    fig.text(0.06, 0.02, caption_text, fontsize=12, va='bottom', ha='left', wrap=True)

    plt.savefig(output_path, dpi=150, bbox_inches='tight', pad_inches=0.4)
    print(f'[V] Plot saved -> {output_path}')
    plt.show()

In [ ]:
def run_simulation(
    k: int = 100,
    seed: int = 2,
    use_predictions: bool = True,
    jsonl_path: str | None = None,
    output_dir: str | None = None,
    weight_mode: str = "additive",
):
    if use_predictions:
        all_cases = load_cases_from_jsonl(
            jsonl_path=jsonl_path, k=k, seed=seed, weight_mode=weight_mode
        )
    else:
        all_cases = generate_cases(k=k if k is not None else 100, seed=seed)

    triage = apply_triage(all_cases)

    phase2_by_random_order = sorted(triage["phase2"], key=lambda case: case.case_id)
    df_random = build_dataframe(phase2_by_random_order, label="RANDOM")
    df_treated = build_dataframe(triage["phase2"], label="TREATED")
    df_phase1 = build_dataframe(triage["phase1"], label="PHASE1", compute_score=False)

    merged = rank_shift_stats(df_random, df_treated)
    metrics = compute_metrics(merged, triage)

    if output_dir is None:
        target_root = RESULTS_DIR
    else:
        target_root = Path(output_dir)

    if use_predictions:
        results_dir = target_root / f"eval_predictions_{k if k else 100}_{seed}"
        prefix = f"{k if k else 100}-{seed}-"
    else:
        results_dir = target_root / f"{k if k is not None else 100}-{seed}"
        prefix = f"{k if k is not None else 100}-{seed}-"

    results_dir.mkdir(parents=True, exist_ok=True)

    df_random.to_csv(results_dir / f"{prefix}random_cases.csv", index=False)
    df_treated.to_csv(results_dir / f"{prefix}treated_cases.csv", index=False)
    df_phase1.to_csv(results_dir / f"{prefix}phase1_acr_d.csv", index=False)
    merged.to_csv(results_dir / f"{prefix}merged_analysis.csv", index=False)

    plot_analysis(merged, triage, output_path=str(results_dir / f"{prefix}analysis.png"))

    return df_random, df_treated, df_phase1, merged, metrics

In [ ]:
# Execute simulation on eval_predictions.jsonl (or synthetic if predictions not available yet)
try:
    df_random, df_treated, df_phase1, merged, metrics = run_simulation(
        k=100, seed=2, use_predictions=True
    )
except FileNotFoundError as e:
    print(f'Warning: {e}')
    print('Running simulation with synthetic cases (use_predictions=False) as fallback demonstration...')
    df_random, df_treated, df_phase1, merged, metrics = run_simulation(
        k=100, seed=2, use_predictions=False
    )

print('\n=================== SUMMARY METRICS ===================')
for metric_name, val in metrics.items():
    print(f'{metric_name:<40}: {val}')

print('\n=================== Top 10 Most Urgent Cases (RANDOM order) ===================')
print(df_random.head(10).to_string(index=False))

print('\n=================== Top 10 Most Urgent Cases (TREATED order) ===================')
print(df_treated.head(10).to_string(index=False))